# **Imports**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as patches
import numpy as np
import os
import datetime
import math
import pickle
from dataclasses import dataclass

!pip install backtrader
%matplotlib inline
import backtrader as bt

# --- General Plotting Setup (run this once in your notebook/script) ---
plt.rcParams['figure.figsize'] = [18, 10] # Default figure size
plt.rcParams['figure.dpi'] = 100 # Default DPI


In [ ]:
print("backtrader:",bt.__version__)
print("numpy",np.__version__)
print("pandas",pd.__version__)

backtrader: 1.9.78.123
numpy 2.0.2
pandas 2.2.2


In [ ]:
%cd /content/
!rm -r crypto-bt-strategy
!git clone https://github.com/ssnsbr/crypto-bt-strategy.git
!ls
%cd crypto-bt-strategy
# !cp -r crypto-bt-strategy/* .
!ls


/content
Cloning into 'crypto-bt-strategy'...
remote: Enumerating objects: 638, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 638 (delta 122), reused 222 (delta 68), pack-reused 357 (from 1)
Receiving objects: 100% (638/638), 10.49 MiB | 9.07 MiB/s, done.
Resolving deltas: 100% (369/369), done.
crypto-bt-strategy  drive  sample_data
/content/crypto-bt-strategy
 analysis	       riskmanagers
 andornot.py	       run.ipynb
 backtrader_extended   run_results
 experiments	       simple_scripts
 __init__.py	       solana_strategies
 main.py	       test.py
 pdma_pandas.py        utils
 pine		      'volab Backtrader_1s_candle_strategy_test_+_git.ipynb'
 raw_indicators


In [ ]:
# %load_ext google.colab.data_table

In [ ]:
from backtrader_extended.sizers.ScalperMartingaleSizer import ScalperMartingaleSizer
from backtrader_extended.strategies.FastScalperStrategy import FastScalperStrategy
from backtrader_extended.strategies.FiboMartingaleStrategy import FiboMartingaleStrategy
from utils.data_utils import *
from utils.plotting_utils import *
from riskmanagers.NoneRiskManagement import NoneRiskManagement
from backtrader_extended.strategies.Base import BaseTradingStrategy

from riskmanagers.ABCRiskManagement import AbstractRiskManagement

from run_results.runner import *
from run_results.runner_utils import *
import backtrader as bt
from utils.utils import format_marketcap, format_price_to_marketcap
from utils.utils import *
from run_results.analys_results import *
from run_results.custom_analyzers import BACounterAnalyzer, CashHistoryAnalyzer, TradeDurationAnalyzer
from backtrader_extended.sizers.FiboMartingaleSizer import FiboMartingaleSizer
from backtrader_extended.strategies import FiboMartingaleStrategy
from run_results.analyse_depth import *
# from utils.bounce_detector import BounceDetector
# from utils.liveliness_tracker import LivelinessTracker
from backtrader_extended.sizers.MartingaleSizer import MartingaleSizer

from backtrader_extended.commissions.CustomSolanaCommission import CustomSolanaCommission
from run_results.runner import calculate_starting_index_time, run_backtest_for_df
from run_results.runner_config import RunConfig
from backtrader_extended.sizers.FiboMartingaleSizer import FiboMartingaleSizer
from backtrader_extended.strategies import FiboMartingaleStrategy
from utils.utils import get_name
from backtrader_extended.strategies.Base_Crypto import BaseCryptoTradingStrategy

from run_results.runner_sol import *


In [ ]:
# !pip install ccxt

# **Data**

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
results_folder='/content/drive/MyDrive/charts/'+"results/"

folder_path_history = '/content/drive/MyDrive/charts/'+"History/"
# List all CSV files
csv_files_h = [folder_path_history+f for f in os.listdir(folder_path_history) if f.endswith('.csv.gz')]
print([f[37:] for f in csv_files_h ])
columns = [ "timestamp",	"Open"	,	"High",	"Low",	"Close"	,	"VolumeTradedinbaseasset",	 "TakerBuyQuoteAssetVolume",	 "TakerBuyBaseAssetVolume",	 "QuoteAssetVolume",	 "Numberoftrades"]
index=-1
try:
    rawdf = pd.read_csv(csv_files_h[index],sep="|",header=None)
    rawdf.columns=columns
    print(f"Successfully read the {csv_files_h[index]}.csv.gz file into a DataFrame!")
    print(rawdf.tail()) # Display the first few rows
    print(rawdf.info()) # Get a summary of the DataFrame
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

['/XRPUSDT.csv.gz', '/BTCUSDT.csv.gz', '/SOLUSDT_11_Aug_2020_to_06_Nov_2025.csv.gz', '/SOLUSDTfrom11Aug2020to23Feb2026.csv.gz', '/SOLUSDT.csv.gz']
Successfully read the /content/drive/MyDrive/charts/History/SOLUSDT.csv.gz.csv.gz file into a DataFrame!
          timestamp   Open   High    Low  Close  VolumeTradedinbaseasset  \
2913488  1772020320  83.01  83.05  83.01  83.04                 5512.365   
2913489  1772020380  83.04  83.08  83.04  83.07                 1979.352   
2913490  1772020440  83.07  83.10  83.06  83.09                  606.996   
2913491  1772020500  83.08  83.11  83.06  83.10                  989.517   
2913492  1772020560  83.11  83.11  83.10  83.10                   63.119   

         TakerBuyQuoteAssetVolume  TakerBuyBaseAssetVolume  QuoteAssetVolume  \
2913488                147490.430                 1776.190        457703.514   
2913489                132832.521                 1599.207        164409.088   
2913490                 45802.103                  

In [ ]:
rawdf.tail()

,timestamp,Open,High,Low,Close,VolumeTradedinbaseasset,TakerBuyQuoteAssetVolume,TakerBuyBaseAssetVolume,QuoteAssetVolume,Numberoftrades
2913488,1772020320,83.01,83.05,83.01,83.04,5512.365,147490.430,1776.190,457703.514,491
2913489,1772020380,83.04,83.08,83.04,83.07,1979.352,132832.521,1599.207,164409.088,451
2913490,1772020440,83.07,83.10,83.06,83.09,606.996,45802.103,551.266,50432.026,257
2913491,1772020500,83.08,83.11,83.06,83.10,989.517,67609.416,813.784,82209.367,291
2913492,1772020560,83.11,83.11,83.10,83.10,63.119,398.263,4.792,5245.237,42


In [ ]:


def updatedf(rawdf):
    # df = rawdf.copy()
    # Slice the last 100,000 rows. Use .copy() to prevent SettingWithCopyWarning later.
    df = rawdf[-600_000:].copy()
    # Assign columns *after* reading
    df["index"]=df.index
    # --- IMPORTANT FIXES START HERE ---
    # 1. Convert the 'timestamp' column to datetime objects
    df['time'] = pd.to_datetime(df['timestamp'], unit='s')

    # 2. Set the 'timestamp' column as the DataFrame's index
    df.set_index('time', inplace=True)
    # Alternatively, if you want to drop the original 'timestamp' column after setting index:
    # df = df.set_index('timestamp') # This returns a new DataFrame

    # Now, df.index *is* a DatetimeIndex, so .minute will work
    df['is_star'] = df['Close'] == df['Open']
    df['is_green'] = df['Close'] > df['Open']
    df['is_red'] = df['Close'] < df['Open']

    df['candle_size'] = df['High'] - df['Low']
    df['candle_body'] = df['Close'] - df['Open']
    # df['candle_body'] = df['candle_body'].Absolute()

    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['day_of_week'] = df.index.day_name()
    df['month'] = df.index.month

    return df

df = updatedf(rawdf)
print(f"Successfully loaded and processed {len(df)} rows.")

Successfully loaded and processed 600000 rows.


In [ ]:

print(f"Successfully loaded and processed {len(df)} rows.")
print("DataFrame Info after processing:")
print(df.info())
print("\nFirst 5 rows (including new columns):")
print(df.tail())

Successfully loaded and processed 600000 rows.
DataFrame Info after processing:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 600000 entries, 2025-01-04 19:57:00 to 2026-02-25 11:56:00
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   timestamp                 600000 non-null  int64  
 1   Open                      600000 non-null  float64
 2   High                      600000 non-null  float64
 3   Low                       600000 non-null  float64
 4   Close                     600000 non-null  float64
 5   VolumeTradedinbaseasset   600000 non-null  float64
 6   TakerBuyQuoteAssetVolume  600000 non-null  float64
 7   TakerBuyBaseAssetVolume   600000 non-null  float64
 8   QuoteAssetVolume          600000 non-null  float64
 9   Numberoftrades            600000 non-null  int64  
 10  index                     600000 non-null  int64  
 11  is_star                   600000 non-null 

In [ ]:

import pandas as pd

def ready_df(df_input, dfsource="auto"):
    """
    Prepare a DataFrame for Backtrader ingestion.
    Works with either:
    - Minimal OHLCV DataFrames with datetime index, or
    - Extended DataFrames (e.g. from Binance) with many extra columns.
    """

    print("Preparing dataframe with size", len(df_input))

    df = df_input.copy()

    # Ensure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')

    # Handle possible column name variations
    col_map = {
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume',
        'VolumeTradedinbaseasset': 'volume',
        'VolumeTradedInBaseAsset': 'volume',
        'QuoteAssetVolume': 'quote_volume',
    }
    df = df.rename(columns=col_map)

    # Add datetime column (Backtrader expects a column, not index)
    df = df.reset_index().rename(columns={'time': 'datetime'})
    if 'timestamp' not in df.columns:
        df['timestamp'] = (df['datetime'].astype('int64') // 10**9)

    # Pick columns relevant to Backtrader
    bt_cols = ['datetime', 'timestamp', 'open', 'high', 'low', 'close', 'volume']
    df_bt = df[[c for c in bt_cols if c in df.columns]]

    # Make sure datetime is datetime dtype
    df_bt['datetime'] = pd.to_datetime(df_bt['datetime'], errors='coerce')

    # Drop NaNs (in case of conversion errors)
    df_bt = df_bt.dropna(subset=['datetime'])

    # Sort by datetime
    df_bt = df_bt.sort_values('datetime').reset_index(drop=True)

    return df_bt




In [ ]:
# ddd= sol_data_ccxt.copy()

In [ ]:
# ddd["datetime"] = ddd.index


In [ ]:
# sol_data_ccxt

# Runner

In [ ]:
import backtrader as bt
def add_to_saves(new_row):
    if new_row["total_trades"] != 0 :
        # 1. Read the existing DataFrame from a CSV
        r_file = results_folder + "solana_results.csv"
        df = pd.read_csv(r_file)
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

        # 4. Save back to CSV (overwrite or create new)
        df.to_csv(r_file, index=False)
        print(len(df))
        print(r_file,"SAVED!")
    else:
        print("NOT SAVED! ZERO TRADES.")
def run_me(strategy_class,df_len=1000,commission=0,sizer_stake=1):
      sizer_params={ "stake":sizer_stake     }
      sizer_class  = bt.sizers.FixedSize
      if True:
        sizer_class  = bt.sizers.PercentSizer
        sizer_params={"percents":10}


      sp={ }

      config = RunConfig()

      data = ready_df(df)[-df_len:]
      # data = ready_df(sol_data_ccxt)
      config.multi_tf=["1m"]

      name, detail = get_name(strategy_class, sp, sizer_class, sizer_params, len(data), config=config)
      results_folder = config.results_folder
      full_save_name = name + "_crypto.csv"
      full_detail_name = "details_" + name + "_crypto_details.txt"
      print(name, detail)
      commission_class=None
      if commission_class is None:
          class CommSOLAxiom(bt.CommissionInfo):
              # 0.005 means 0.5% of the operation value
              # 0.0001 means 0.01% of the operation value
              # params = dict(commission=0.0001)
              params = dict(commission=commission)

          commission_class = CommSOLAxiom

      all_results_df, all_cerebros_objects, all_portfolio_histories = run_crypto_df(data,
                                                                                    "sol",
                                                                                    sizer_class=sizer_class,
                                                                                    strategy_class=strategy_class,
                                                                                    strategy_params=sp,
                                                                                    sizer_params=sizer_params,
                                                                                    config=config,
                                                                                    commission_class=commission_class,
                                                                                    )
      all_results_df["strategy_class"] = strategy_class.__name__
      all_results_df["len_data"] = df_len
      all_results_df["commission_class"] = commission_class.__name__
      all_results_df["commission"] = commission
      print("DONE FOR ",strategy_class.__name__)
      add_to_saves(all_results_df)
      return all_results_df, all_cerebros_objects, all_portfolio_histories


# run_me(SOL_TrendAware_MeanReversion_RSI)
# strategy_class = SOL_TrendAware_MeanReversion_RSI


In [ ]:
import json
import pickle
import os

def ready_df(df_input, dfsource="auto"):
    """
    Prepare a DataFrame for Backtrader ingestion.
    Keeps both 'datetime' (datetime type) and 'timestamp' (numeric) columns.
    Handles both simple OHLCV and extended Binance-style data.
    """

    print("Preparing dataframe with size", len(df_input))

    df = df_input.copy()

    # Ensure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')

    # Normalize column names (case-insensitive)
    col_map = {
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume',
        'VolumeTradedinbaseasset': 'volume',
        'VolumeTradedInBaseAsset': 'volume',
        'QuoteAssetVolume': 'quote_volume',
    }
    df = df.rename(columns=col_map)

    # Reset index → create 'datetime' column
    df = df.reset_index().rename(columns={'time': 'datetime'})

    # If timestamp is missing, create it from datetime
    if 'timestamp' not in df.columns:
        df['timestamp'] = (df['datetime'].astype('int64') // 10**9)

    # Keep only Backtrader-relevant + timestamp
    bt_cols = ['datetime', 'timestamp', 'open', 'high', 'low', 'close', 'volume']
    df_bt = df[[c for c in bt_cols if c in df.columns]]

    # Ensure datetime is proper dtype
    df_bt['datetime'] = pd.to_datetime(df_bt['datetime'], errors='coerce')

    # Clean up NaNs and sort
    df_bt = df_bt.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)

    return df_bt


In [ ]:
e / 0

NameError: name 'e' is not defined

# Strategy

In [ ]:
from collections import deque
from typing import Tuple

from dataclasses import dataclass
from typing import Type, Tuple
from abc import ABC, abstractmethod


class Indicator(ABC):
    def __init__(self, key: str):
        self._key = key

    def key(self) -> str:
        return self._key

    @abstractmethod
    def update(self, candle ):
        pass

    @abstractmethod
    def value(self):
        pass

    def values(self) -> dict:
        """
        Override to expose multiple outputs.
        Default wraps single value() for backwards compatibility.
        """
        return {"value": self.value()}

    def reset(self):
        pass

    def ready(self) -> bool:
        return True

    def get_state(self) -> dict:
        """
        Return internal state needed to restore indicator.
        Must be JSON-serializable.
        """
        return {}

    def set_state(self, state: dict):
        """
        Restore indicator from state produced by get_state().
        """
        pass


class DEMA(Indicator):
    """
    Double Exponential Moving Average: 2*EMA(src, len) - EMA(EMA(src, len), len)
    Streaming implementation.
    """

    def __init__(self, period: int):
        self.period = period
        self.k = 2.0 / (period + 1)

        self._ema1 = None
        self._ema2 = None
        self._count = 0
        self._value = None

    def update(self, candle):
        self.update_value(candle.close)

    def update_value(self, price: float):
        if self._ema1 is None:
            self._ema1 = price
            self._ema2 = price
            self._count = 1
        else:
            self._ema1 = price * self.k + self._ema1 * (1 - self.k)
            self._ema2 = self._ema1 * self.k + self._ema2 * (1 - self.k)
            self._count += 1

        self._value = 2 * self._ema1 - self._ema2

    def value(self):
        return self._value

    def ready(self) -> bool:
        return self._count >= self.period

    def get_state(self) -> dict:
        return {
            "ema1": self._ema1,
            "ema2": self._ema2,
            "count": self._count,
            "value": self._value,
        }

    def set_state(self, state: dict):
        self._ema1 = state["ema1"]
        self._ema2 = state["ema2"]
        self._count = state["count"]
        self._value = state["value"]


class BarsSinceCross(Indicator):
    """
    Streaming equivalent of:
        bars_i = ta.barssince(ta.cross(close, ma))
        z_i    = -1 if ma > close else 1
    Tracks bars since last cross + sign of position.
    Optionally normalises by MA period (matching the document code).
    """

    def __init__(self, dema: DEMA, period: int, useZ: bool = True, normalize_by_period: bool = True):
        self.dema = dema
        self.period = period
        self.useZ = useZ
        self.normalize_by_period = normalize_by_period

        self.bars_since: int = 0
        self._value: float = 0.0
        self._prev_diff = None

    def update(self, candle):
        close = candle.close
        ma_val = self.dema.value()

        if ma_val is None:
            return

        diff = close - ma_val

        self.bars_since += 1

        # cross detection: sign flip or exact touch
        if self._prev_diff is not None:
            if diff == 0 or (diff * self._prev_diff < 0):
                self.bars_since = 0

        self._prev_diff = diff

        raw = self.bars_since / self.period if self.normalize_by_period else float(self.bars_since)

        if self.useZ:
            z = -1 if ma_val > close else 1
            self._value = raw * z
        else:
            self._value = raw

    def value(self) -> float:
        return self._value

    def ready(self) -> bool:
        return self.dema.ready()

    def get_state(self) -> dict:
        return {
            "bars_since": self.bars_since,
            "_value": self._value,
            "_prev_diff": self._prev_diff,
        }

    def set_state(self, state: dict):
        self.bars_since = state["bars_since"]
        self._value = state["_value"]
        self._prev_diff = state["_prev_diff"]


class TimeDistance(Indicator):
    """
    Streaming translation of the Pine Script ind.next() logic.

    Uses 5 DEMA MAs. For each bar:
      1. Average (optionally signed) bars-since-cross across all 5 MAs.
      2. Smooth with EMA(smooth).
      3. Z-score normalize with SMA/StdDev(normalizing).
      4. Signal line: EMA(normalized, signaling).

    Public attributes after .update(candle):
        .time_norm      – normalized oscillator value
        .time_signal    – signal line
        .slope_up       – bool: time_norm > time_signal
        .slope_down     – bool: time_norm < time_signal
    """

    MA_LENGTHS = (21, 50, 100, 150, 200)  # defaults matching typical Pine usage

    def __init__(
        self,
        ma_lengths: Tuple[int, ...] = MA_LENGTHS,
        smooth: int = 23,
        useZ: bool = True,
        normalizing: int = 200,
        signaling: int = 15,
        # normalize_by_period: bool = False,
    ):
        self.smooth = smooth
        self.useZ = useZ
        self.normalizing = normalizing
        self.signaling = signaling

        # 5 DEMAs + their bars-since-cross trackers
        self.demas = [DEMA(length) for length in ma_lengths]
        self.bars_inds = [
            BarsSinceCross(dema, length, useZ)
            for dema, length in zip(self.demas, ma_lengths)
        ]

        # Smoothing EMA
        self._avg_ema = _StreamingEMA(smooth)

        # Normalisation buffers (rolling SMA + StdDev)
        self._norm_buf: deque = deque(maxlen=normalizing)

        # Signal EMA
        self._signal_ema = _StreamingEMA(signaling)

        # Outputs
        self.time_norm: float | None = None
        self.time_signal: float | None = None
        self.slope_up: bool = False
        self.slope_down: bool = False

    # ------------------------------------------------------------------
    def update(self, candle):
        # Step 1 – update all DEMAs, then bars-since trackers
        for dema in self.demas:
            dema.update(candle)

        for bi in self.bars_inds:
            bi.update(candle)

        # Need all DEMAs warm before proceeding
        if not all(d.ready() for d in self.demas):
            return

        # Step 2 – average bars values  (mirrors Pine: sum/5)
        avg = sum(bi.value() for bi in self.bars_inds) / len(self.bars_inds)

        # Step 3 – smooth
        self._avg_ema.update(avg)
        if not self._avg_ema.ready():
            return
        smoothed = self._avg_ema.value()

        # Step 4 – rolling z-score
        self._norm_buf.append(smoothed)
        if len(self._norm_buf) < self.normalizing:
            return

        mean = sum(self._norm_buf) / self.normalizing
        variance = sum((x - mean) ** 2 for x in self._norm_buf) / self.normalizing
        std = variance ** 0.5

        if std == 0:
            return

        self.time_norm = (smoothed - mean) / std

        # Step 5 – signal line
        self._signal_ema.update(self.time_norm)
        if self._signal_ema.ready():
            self.time_signal = self._signal_ema.value()
            self.slope_up = self.time_norm > self.time_signal
            self.slope_down = self.time_norm < self.time_signal

    def value(self):
        return self.time_norm

    def values(self) -> dict:
        return {
            "value": self.time_norm,        # keeps .value() contract
            "signal": self.time_signal,
            "slope_up": self.slope_up,
            "slope_down": self.slope_down,
        }

    def ready(self) -> bool:
        return self.time_signal is not None

    # ------------------------------------------------------------------
    def get_state(self) -> dict:
        return {
            "demas": [d.get_state() for d in self.demas],
            "bars_inds": [b.get_state() for b in self.bars_inds],
            "avg_ema": self._avg_ema.get_state(),
            "norm_buf": list(self._norm_buf),
            "signal_ema": self._signal_ema.get_state(),
            "time_norm": self.time_norm,
            "time_signal": self.time_signal,
            "slope_up": self.slope_up,
            "slope_down": self.slope_down,
        }

    def set_state(self, state: dict):
        for d, s in zip(self.demas, state["demas"]):
            d.set_state(s)
        for b, s in zip(self.bars_inds, state["bars_inds"]):
            b.set_state(s)
        self._avg_ema.set_state(state["avg_ema"])
        self._norm_buf.clear()
        self._norm_buf.extend(state["norm_buf"])
        self._signal_ema.set_state(state["signal_ema"])
        self.time_norm = state["time_norm"]
        self.time_signal = state["time_signal"]
        self.slope_up = state["slope_up"]
        self.slope_down = state["slope_down"]


# ---------------------------------------------------------------------------
# Minimal self-contained EMA used internally (no candle dependency)
# ---------------------------------------------------------------------------

class _StreamingEMA:
    """Lightweight EMA that accepts raw float values (not candles)."""

    def __init__(self, period: int):
        self.period = period
        self.k = 2.0 / (period + 1)
        self._val: float | None = None
        self._count: int = 0

    def update(self, price: float):
        if self._val is None:
            self._val = price
        else:
            self._val = price * self.k + self._val * (1 - self.k)
        self._count += 1

    def value(self) -> float | None:
        return self._val

    def ready(self) -> bool:
        return self._count >= self.period

    def get_state(self) -> dict:
        return {"val": self._val, "count": self._count}

    def set_state(self, state: dict):
        self._val = state["val"]
        self._count = state["count"]


# Usage
# ind = TimeDistance(
#     ma_lengths=(21, 50, 100, 150, 200),
#     smooth=23,
#     useZ=True,
#     normalizing=200,
#     signaling=15,
# )

# for candle in live_candles:
#     ind.update(candle)
#     if ind.ready():
#         print(ind.time_norm, ind.time_signal, ind.slope_up)


# S

In [ ]:

class PDMAStrategy(BaseCryptoTradingStrategy):
    """PDMA / ma_b Strategy: Price from Last Touch"""
    params = (
        ('ma_types', ['dema', 'dema', 'dema', 'dema', 'dema']),  # MA types
        ('ma_periods', [50, 100, 150, 200, 250]),  # MA periods
        ('src', 'hlc3'),  # Source data
        ('smooth', 23),
        ('tpPerc', 10.0),
        ('slPerc', 10.0),
        ('reverse', False),
        ('useZtime', True),
        ('lengthMA', 34),
        ('lengthSignal', 9),
        ('normalize_len', 200),
        ('signal_len', 15),
        ('ema_filter',[True,"dema",360])
    )

    def __init__(self):
        super().__init__()
        # custom indicator
        self.ind = TimeDistance(
          ma_lengths=(50, 100, 150, 200, 250),
          smooth=23,
          useZ=True,
          normalizing=200,
          signaling=15,
        )
        if self.params.ema_filter[0]:
            # custom indicator
            period=self.p.ema_filter[2]
            if self.params.ema_filter[1] == "dema":
                self.ema = DEMA( period=period)
            if self.params.ema_filter[1] == "ema":
                self.ema = bt.indicators.EMA(self.datas[0], period=period)
            else:
                self.ema = bt.indicators.EMA(self.datas[0], period=period)
        print("init done.")
    def _execute_trading_logic(self):
        price = self.datas[0].close[0]
        self.ind.update(self.datas[0]) # update gets a candle object which should have object.close
        # if self.params.ema_filter[0]:
        #     self.ema.update(self.datas[0])
        #     ema_value = self.ema.values()["value"]
        ema_value = self.ema[0]

        # Get indicator values


        ma_b_norm_signal = self.ind.values()["signal"]
        slope_ma_b_up =  self.ind.values()["slope_up"]
        slope_ma_b_down =  self.ind.values()["slope_down"]
        # slope_ma_b_up = ma_b_norm > ma_b_norm_signal
        # slope_ma_b_down = ma_b_norm < ma_b_norm_signal


        if self.params.reverse:
            longCond = slope_ma_b_down and ma_b_norm_signal > 0
            shortCond = slope_ma_b_up and ma_b_norm_signal < 0
        else:
            longCond = slope_ma_b_up and ma_b_norm_signal < 0
            shortCond = slope_ma_b_down and ma_b_norm_signal > 0
        # print("longCond",longCond,"shortCond:",shortCond,"Signal:",ma_b_norm_signal,"Up:",slope_ma_b_down,"Down:",slope_ma_b_up)

        # Execute trades




        # =========================
        # NO POSITION
        # =========================
        if not self.position:
            if longCond:
                if self.params.ema_filter[0] :
                  if price > ema_value:
                    self.buy()
                else:
                  self.buy()

            elif shortCond:
                if self.params.ema_filter[0] :
                  if price < ema_value:
                    self.sell()
                else:
                  self.sell()

        # =========================
        # POSITION OPEN
        # =========================
        else:
            entry_price = self.position.price

            tpPerc = self.params.tpPerc / 100.0
            slPerc = self.params.slPerc / 100.0

            # ---------- LONG ----------
            if self.position.size > 0:
                tp_price = entry_price * (1 + tpPerc)
                sl_price = entry_price * (1 - slPerc)

                # TP / SL check
                if price >= tp_price or price <= sl_price:
                    self.close()
                    return

                # Opposite signal
                if shortCond:
                    self.close()
                    # self.sell()
                    return

            # ---------- SHORT ----------
            elif self.position.size < 0:
                tp_price = entry_price * (1 - tpPerc)
                sl_price = entry_price * (1 + slPerc)

                # TP / SL check
                if price <= tp_price or price >= sl_price:
                    self.close()
                    return

                # Opposite signal
                if longCond:
                    self.close()
                    # self.buy()
                    return



In [ ]:
df[-13600:].head()

,timestamp,Open,High,Low,Close,VolumeTradedinbaseasset,TakerBuyQuoteAssetVolume,TakerBuyBaseAssetVolume,QuoteAssetVolume,Numberoftrades,...,is_star,is_green,is_red,candle_size,candle_body,minute,hour,day,day_of_week,month
time,,,,,,,,,,,,,,,,,,,,,
2026-02-16 01:17:00,1771204620,86.35,86.39,86.34,86.35,629.682,12602.152,145.907,54381.287,330,...,True,False,False,0.05,0.00,17,1,16,Monday,2
2026-02-16 01:18:00,1771204680,86.36,86.38,86.33,86.34,391.988,5717.200,66.199,33851.972,203,...,False,False,True,0.05,-0.02,18,1,16,Monday,2
2026-02-16 01:19:00,1771204740,86.33,86.34,86.26,86.26,474.391,5119.337,59.303,40941.544,310,...,False,False,True,0.08,-0.07,19,1,16,Monday,2
2026-02-16 01:20:00,1771204800,86.26,86.26,86.15,86.17,747.503,18345.705,212.886,64423.131,446,...,False,False,True,0.11,-0.09,20,1,16,Monday,2
2026-02-16 01:21:00,1771204860,86.18,86.18,86.13,86.17,2574.593,64480.192,748.425,221797.864,555,...,False,False,True,0.05,-0.01,21,1,16,Monday,2


In [ ]:
# from run_results.runner_sol import run_crypto_df
l=13600
l=3600

print(l/(24*60),"days.")
all_results_df, all_cerebros_objects, all_portfolio_histories=run_me(PDMAStrategy,l,sizer_stake=0.5)


2.5 days.
Preparing dataframe with size 600000


/tmp/ipython-input-10966/2033167742.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bt['datetime'] = pd.to_datetime(df_bt['datetime'], errors='coerce')


PDMAStrategy_PercentSizer_2026-02-26_12:51:41_len-3600 {'strategy': 'PDMAStrategy', 'strategy_params': {}, 'sizer': 'PercentSizer', 'sizer_params': {'percents': 10}, 'time': '2026-02-26_12:51:41', 'len': 3600, 'run_config': {'results_folder': '/content/drive/MyDrive/charts/results/', 'cash': 100, 'mcap': True, 'multi_tf': ['1m'], 'after_ath': False, 'min_start_minutes_to_wait': 30, 'randomize_start_margin': True, 'df_end_margin': -1, 'max_start_margin': 100, 'min_start_margin': 5, 'cerebro_runonce': True}}

******************** Running backtest for sol ( Len: 3600) ********************
Start margin: 0
[RUN] Strategy: PDMAStrategy, Params: {}
Adding base data: 1m 1
[RUN] Sizer: PercentSizer, Params: {'percents': 10}
[RUN] Not in MCAP mode. Cash: 100.00
[RUN] Commission: CommSOLAxiom
[RUN] Adding Analyzers and Observers.
[RUN] Starting backtest for sol - Initial Portfolio Value: 100.00
[RUN] Cerebro Starting.
Base Trading Strategy Initialized
init done.
[Strategy] [PDMAStrategy] Index 0 

In [ ]:
usingdf=df.copy()[-50_000:]
usingdf["high"]=usingdf["High"]
usingdf["low"]=usingdf["Low"]
usingdf["close"]=usingdf["Close"]
usingdf["open"]=usingdf["Open"]

In [ ]:
w

In [ ]:
# --- (Previous code for data loading and feature creation: is_green, is_red, is_star, minute_of_day, day_of_week) ---

# Now, call the new function for 'is_green' candles
analyze_by_day_and_minute(df, 'is_green', 'Green')

# You can repeat for 'is_red' or 'is_star' if you wish, but be aware of the many plots!
# analyze_by_day_and_minute(df, 'is_red', 'Red')
# analyze_by_day_and_minute(df, 'is_star', 'Doji/Indecision')

In [ ]:
run_me(SOL_Trend_3MA,10_000)
rdf = pd.read_csv(results_folder + "solana_results.csv")
rdf["finalpnl"] = rdf["final_value"] / rdf["start_value"]
rdf=rdf.drop_duplicates(subset=['profit_factor'], keep='last')

rdf[["finalpnl","strategy_class","win_rate","profit_factor","risk_reward_ratio","risk_reward_str","len_data"]].sort_values(["strategy_class","profit_factor"])

# **Results**

In [ ]:

best_coin_name = all_results_df["coin"]
# best_coin_name="ars_ABPmWi"
print(f"\n--- Plotting best performing coin: {best_coin_name} ---")
plot_single_backtest(all_cerebros_objects , title=f"Backtest for {best_coin_name}")

# You can also manually select a coin to plot, e.g.:
# plot_single_backtest(all_cerebros_objects['coin_0'], title="Backtest for coin_0")

In [ ]:
all_results_df, all_cerebros_objects, all_portfolio_histories = run_me(TestStrategy, 2000)


In [ ]:
results = all_cerebros_objects.run()
strat = results[0]
pivots=strat.results["s"]
p_list=list(pivots)

data = pd.DataFrame(strat.results["l"], columns=['datetime', 'close', 'trend', 'pivot'])
data.set_index('datetime', inplace=True)
data


In [ ]:
pivots

In [ ]:
import matplotlib.pyplot as plt

def draw_result(res_df):
    """
    Draws price with color-coded trend and pivot labels (HH, HL, LH, LL).
    """
    plt.figure(figsize=(12, 6))
    plt.plot(res_df['close'], color='black', linewidth=1, label='Price')

    # Color segments by trend
    up = res_df[res_df['trend'] > 0]
    down = res_df[res_df['trend'] < 0]
    flat = res_df[res_df['trend'] == 0]

    plt.scatter(up.index, up['close'], color='lime', label='Uptrend', s=5)
    plt.scatter(down.index, down['close'], color='red', label='Downtrend', s=5)
    plt.scatter(flat.index, flat['close'], color='gray', label='Neutral', s=5)



    # p_list   is [(175.16, 'LL'), (179.08, 'LL'), (173.33, 'LH')]


    # === Draw pivot labels ===

    # === If pivot list exists, draw labels ===
    # === Draw pivot markers and labels ===
    for t, price, label in p_list:
        color = (
            'lime' if 'H' in label and label != 'LH' else
            'red' if 'L' in label and label != 'HL' else
            'lime'
        )
        plt.scatter(t, price, color=color, s=5, edgecolors='black', zorder=5)
        plt.text(
            t, price,
            f'{label}',
            fontsize=8,
            ha='center',
            va='bottom' if 'H' in label else 'top',
            color=color,
            weight='bold',
            zorder=6,
        )

    plt.title("HH/LL Trend Structure Detection")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
draw_result(data)

In [ ]:
import matplotlib.pyplot as plt

def draw_result(res_df):
    """
    Draws price with color-coded trend:
      - Green for uptrend
      - Red for downtrend
      - Gray for neutral
    """
    plt.figure(figsize=(12, 6))
    plt.plot(res_df['close'], color='black', linewidth=1, label='Price')

    # Color segments by trend
    up = res_df[res_df['trend'] > 0]
    down = res_df[res_df['trend'] < 0]
    flat = res_df[res_df['trend'] == 0]

    plt.scatter(up.index, up['close'], color='lime', label='Uptrend', s=10)
    plt.scatter(down.index, down['close'], color='red', label='Downtrend', s=10)
    plt.scatter(flat.index, flat['close'], color='gray', label='Neutral', s=10)

    plt.title("HH/LL Trend Structure Detection")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
draw_result(data)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
_ = plot_single_backtest(all_cerebros_objects , title=f"Backtest for")
plt.show()



In [ ]:
all_cerebros_objects.plot(style="candlestick")
plt.show()



In [ ]:
import numpy as np
import pandas as pd


# ==============================
# === LOAD YOUR DATA FIRST ====
# ==============================
# usingdf must contain:
# ['open','high','low','close','volume']
# index should be datetime

# Example:
# usingdf = pd.read_csv("sol.csv", parse_dates=True, index_col=0)


# ==============================
# === INDICATOR FUNCTIONS ======
# ==============================

def ema(series, length):
    return series.ewm(span=length, adjust=False).mean()


def dema(series, length):
    e1 = ema(series, length)
    e2 = ema(e1, length)
    return 2 * e1 - e2


def cross(series1, series2):
    return (
        ((series1 > series2) & (series1.shift(1) <= series2.shift(1))) |
        ((series1 < series2) & (series1.shift(1) >= series2.shift(1)))
    )


def bars_since(condition):
    idx = np.where(condition, np.arange(len(condition)), np.nan)
    idx = pd.Series(idx, index=condition.index).ffill()
    return pd.Series(np.arange(len(condition)), index=condition.index) - idx


# ==============================
# === STRATEGY PARAMETERS ======
# ==============================

base_ma = 50
multipliers = [1, 2, 3, 4, 5]
smooth = 23
normalize_len = 200
signal_len = 15
tpPerc = 0.01     # 1%
slPerc = 0.01     # 1%
use_tp = True
use_sl = True


# ==============================
# === BUILD INDICATORS =========
# ==============================

usingdf["hlc3"] = (usingdf["high"] + usingdf["low"] + usingdf["close"]) / 3

ma_periods = [base_ma * m for m in multipliers]

bars_list = []

for p in ma_periods:
    ma = dema(usingdf["hlc3"], p)
    usingdf[f"ma_{p}"] = ma

    c = cross(usingdf["close"], ma)
    bars = bars_since(c)

    bars_norm = bars / p
    bars_list.append(bars_norm)

# Average like Pine
usingdf["ma_b_raw"] = sum(bars_list) / len(bars_list)

# Smooth
usingdf["ma_b_smooth"] = ema(usingdf["ma_b_raw"], smooth)

# Z-score normalization
rolling_mean = usingdf["ma_b_smooth"].rolling(normalize_len).mean()
rolling_std = usingdf["ma_b_smooth"].rolling(normalize_len).std()

usingdf["ma_b_norm"] = (usingdf["ma_b_smooth"] - rolling_mean) / rolling_std

# Signal line
usingdf["ma_b_signal"] = ema(usingdf["ma_b_norm"], signal_len)


# ==============================
# === STRATEGY LOGIC ===========
# ==============================

usingdf["slope_up"] = usingdf["ma_b_norm"] > usingdf["ma_b_signal"]
usingdf["slope_down"] = usingdf["ma_b_norm"] < usingdf["ma_b_signal"]

usingdf["longCond"] = usingdf["slope_down"] & (usingdf["ma_b_signal"] > 0)
usingdf["shortCond"] = usingdf["slope_up"] & (usingdf["ma_b_signal"] < 0)


# ==============================
# === POSITION GENERATION ======
# ==============================

usingdf["position"] = 0

usingdf.loc[usingdf["longCond"], "position"] = 1
usingdf.loc[usingdf["shortCond"], "position"] = -1

# Hold position until opposite signal
usingdf["position"] = usingdf["position"].replace(0, np.nan).ffill().fillna(0)


# ==============================
# === SIMPLE BACKTEST ==========
# ==============================

usingdf["returns"] = usingdf["close"].pct_change()

usingdf["strategy_returns"] = usingdf["position"].shift(1) * usingdf["returns"]

# Optional TP/SL (vectorized approximation)
if use_tp or use_sl:
    entry_price = usingdf["close"].where(
        (usingdf["position"] != usingdf["position"].shift(1))
    ).ffill()

    if use_tp:
        long_tp = entry_price * (1 + tpPerc)
        short_tp = entry_price * (1 - tpPerc)

    if use_sl:
        long_sl = entry_price * (1 - slPerc)
        short_sl = entry_price * (1 + slPerc)

    long_exit = False
    short_exit = False

    if use_tp and use_sl:
        long_exit = (usingdf["high"] >= long_tp) | (usingdf["low"] <= long_sl)
        short_exit = (usingdf["low"] <= short_tp) | (usingdf["high"] >= short_sl)

    elif use_tp:
        long_exit = usingdf["high"] >= long_tp
        short_exit = usingdf["low"] <= short_tp

    elif use_sl:
        long_exit = usingdf["low"] <= long_sl
        short_exit = usingdf["high"] >= short_sl

    exit_mask = (
        ((usingdf["position"] == 1) & long_exit) |
        ((usingdf["position"] == -1) & short_exit)
    )

    usingdf.loc[exit_mask, "position"] = 0
    usingdf["position"] = usingdf["position"].replace(0, np.nan).ffill().fillna(0)

    usingdf["strategy_returns"] = usingdf["position"].shift(1) * usingdf["returns"]

# ==========================================
# === TRADE EXTRACTION =====================
# ==========================================
# ==========================================
# === TRADE EXTRACTION (FIXED) ============
# ==========================================

usingdf["pos_change"] = usingdf["position"].diff()

# Entry when position changes from 0 → 1 or 0 → -1
entries = usingdf[usingdf["pos_change"] != 0].copy()

trades = []

current_position = 0
entry_price = None

for idx, row in entries.iterrows():

    new_position = row["position"]

    # If opening new trade
    if current_position == 0 and new_position != 0:
        current_position = new_position
        entry_price = row["close"]

    # If flipping or closing
    elif current_position != 0:

        exit_price = row["close"]

        if current_position == 1:
            pnl = (exit_price - entry_price) / entry_price
        else:
            pnl = (entry_price - exit_price) / entry_price

        trades.append(pnl)

        # start new trade if flipped
        if new_position != 0:
            entry_price = exit_price
            current_position = new_position
        else:
            current_position = 0
            entry_price = None

trades = np.array(trades)
# ==========================================
# === PERFORMANCE METRICS ==================
# ==========================================

if len(trades) > 0:

    wins = trades[trades > 0]
    losses = trades[trades < 0]

    win_rate = len(wins) / len(trades)

    gross_profit = wins.sum()
    gross_loss = abs(losses.sum())

    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf

    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = abs(losses.mean()) if len(losses) > 0 else 0

    risk_reward = avg_win / avg_loss if avg_loss != 0 else np.inf

    expectancy = (win_rate * avg_win) - ((1 - win_rate) * avg_loss)

    print("Trades:", len(trades))
    print("Win Rate:", round(win_rate * 100, 2), "%")
    print("Profit Factor:", round(profit_factor, 2))
    print("Avg Win:", round(avg_win * 100, 3), "%")
    print("Avg Loss:", round(avg_loss * 100, 3), "%")
    print("Risk/Reward:", round(risk_reward, 2))
    print("Expectancy per trade:", round(expectancy * 100, 3), "%")

else:
    print("No trades found.")

# ==============================
# === PERFORMANCE METRICS ======
# ==============================

usingdf["equity"] = (1 + usingdf["strategy_returns"]).cumprod()

total_return = usingdf["equity"].iloc[-1] - 1
sharpe = (
    usingdf["strategy_returns"].mean() /
    usingdf["strategy_returns"].std()
) * np.sqrt(365 * 24)  # adjust for timeframe

max_dd = (
    usingdf["equity"] /
    usingdf["equity"].cummax() - 1
).min()

print("Total Return:", round(total_return * 100, 2), "%")
print("Sharpe:", round(sharpe, 2))
print("Max Drawdown:", round(max_dd * 100, 2), "%")
